# Agente explicador de Regras do Futebol com RAG (Langchain)

In [1]:
# !pip install chromadb --quiet
# !pip install --upgrade langchain langchain-community langchain-openai langchain-core pydantic chromadb pypdf --quiet

In [2]:
import os 

# Loader de documentos em PDF
from langchain_community.document_loaders import PyPDFLoader

#Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_openai import OpenAIEmbeddings 

# Banco Vetorial
from langchain_community.vectorstores import Chroma

#LLM
from langchain_openai import ChatOpenAI

# Cadeia RAG
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

c:\Users\Usuario\Documents\Estudos\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


ModuleNotFoundError: No module named 'langchain.chains'

## Definição do problema 
LLMs possuem conhecimento estático e podem alucinar. O objetivo aqui é garantir respostas confiáveis, conectando o modelo a
documentos oficiais sobre regras do futebol

In [ ]:
# Caminho(s) do(s) arquivo(s)
caminho_pdf = r'C:\Users\Usuario\Documents\Estudos\ALURA\08_arquitetura_rag_com_llms\raw\regras_futebol.pdf'

#Carrega o PDF
loader = PyPDFLoader(caminho_pdf)
documents = loader.load()

# Quantidade de páginas carregadas
len(documents)

232

## Preparação do documentos

In [ ]:
# Divide os documentos em chunks menores 
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents=documents)
len(chunks)

654

## Embeddings e Banco Vetorial

In [ ]:
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

# Inicializa embeddings
embeddings = OpenAIEmbeddings(model='text-embedding-3-small', openai_api_key=api_key)

# Cria o banco vetorial
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory='./chroma_regras_futebol')

## Recuperação de contexto (Retriever)
O retriever busca os trechos mais relevantes para cada pergunta do usuário

In [ ]:
# Cria o retriever 
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k':3})

## Integração com LLM 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
# Inicializar o modelo
llm = ChatOpenAI(model='gpt-4o-mini', api_key=api_key)

# Cria a cadeia RAG
system_prompt = (
    "Você é um assistente especialista nas regras oficiais do futebol. "
    "Use os seguintes pedaços de contexto recuperado para responder à pergunta. "
    "Se você não souber a resposta, diga que não sabe.\n\n"
    "Contexto: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

cadeia_documentos = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(retriever, cadeia_documentos)


In [ ]:
# !pip install "pydantic<2.10"

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [39 lines of output]
      Checking for Rust toolchain....
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp314-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\Usuario\AppData\Local\puccinialin\puccinialin\Cache
      
      Installing rust to C:\Users\Usuario\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\Usuario\.rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      info: profile set to minimal
      info: default host triple is x86_64-pc-windows-msvc
      info: syncing channel updates for stable-x86_64-pc-windows-msvc
      info: 

In [ ]:
pergunta = 'Um jogador pode usar a mão para marcar um gol?'

# Executa a pergunta no agente RAG usando .invoke() e passando um dicionário
resposta = qa_chain.invoke({"input": pergunta})

print('Pergunta:')
print(pergunta)

print('\nResposta do Agente:')
# A chave de resposta agora é 'answer'
print(resposta['answer'])

print('\nTrechos utilizados como contexto:\n')

# Os documentos recuperados ficam na chave 'context'
for i, doc in enumerate(resposta['context'], start=1):
    print(f'Trecho {i}')
    # Usando aspas duplas por fora para não dar conflito com as aspas simples de dentro
    print(f"Fonte: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print('Conteúdo:')
    print(doc.page_content)
    print('\n')

ValidationError: 1 validation error for SystemMessage
additional_kwargs
  Input should be a valid dictionary [type=dict_type, input_value=FieldInfo(default=Pydanti...class 'dict'>, extra={}), input_type=FieldInfo]
    For further information visit https://errors.pydantic.dev/2.12/v/dict_type